In [3]:
import pandas as pd

# Carrega a base de colaboradores
colaboradores = pd.read_csv('colaboradores.csv')

# Agrupa por empresa
grupos = colaboradores.groupby('empresa')

# Separa colaboradores por empresa
df_vertex = grupos.get_group('Vertex Logística')
df_sopro = grupos.get_group('Sopro Alimentos')
df_aurora = grupos.get_group('Aurora Clínicas')

# display(df_vertex)
# display(df_sopro)
# display(df_aurora)

In [14]:
# descobrir quem ainda está na empresa e quem já saiu

# Data de referência: agosto de 2026
mes_referencia = pd.Period('2026-08', freq='M')

# Devolve o mês de referência atual (mês anterior ao mês corrente)
# mes_referencia = pd.Period.now(freq='M')-1

def analisar_desligados(df, nome_empresa):
    df = df.copy()
    # Converte a coluna de data de desligamento para datetime, tratando erros
    df['data_desligamento'] = pd.to_datetime(df['data_desligamento'], errors='coerce')
    
    # Quem já saiu (independente do mês)
    desligados = df[df['status'] == 'desligado']
    
    # Regra: mantém na base quem saiu no mês de referência (ago/2026)
    saiu_no_mes_referencia = desligados['data_desligamento'].dt.to_period('M') == mes_referencia
    
    # Base final: todos os admitidos + desligados do mês de referência
    df_base_atualizada = df[(df['status'] == 'admitido') | ((df['status'] == 'desligado') & saiu_no_mes_referencia)]
    
    # Quem sai definitivamente da base (desligados antes do mês de referência)
    df_removidos = desligados[~saiu_no_mes_referencia]

    print(f"\n=== {nome_empresa} — Removidos do mês de referência ===")
    print(df_removidos[['nome', 'data_desligamento']].to_string(index=False))
    
    return df_base_atualizada, df_removidos

df_vertex_base, df_vertex_removidos = analisar_desligados(df_vertex, 'Vertex Logística')
df_sopro_base, df_sopro_removidos     = analisar_desligados(df_sopro, 'Sopro Alimentos')
df_aurora_base, df_aurora_removidos   = analisar_desligados(df_aurora, 'Aurora Clínicas')



=== Vertex Logística — Removidos do mês de referência ===
                   nome data_desligamento
     Ana Klein Oliveira        2026-07-18
Thiago Queiroz Oliveira        2026-07-29

=== Sopro Alimentos — Removidos do mês de referência ===
Empty DataFrame
Columns: [nome, data_desligamento]
Index: []

=== Aurora Clínicas — Removidos do mês de referência ===
Empty DataFrame
Columns: [nome, data_desligamento]
Index: []


In [12]:
# exporta o resultado para um arquivo CSV
df_vertex_base.to_csv("colaboradores_vertex.csv", index=False)
df_sopro_base.to_csv("colaboradores_sopro.csv", index=False)
df_aurora_base.to_csv("colaboradores_aurora.csv", index=False)